# 🇧🇩 Bangla Word Sense Disambiguation (WSD) — FULLY FIXED
### Production-Quality BanglaBERT Fine-Tuning with [TGT] Markers & Word-Specific Labels

---

## ❌ WHY YOUR PREVIOUS MODEL FAILED — ROOT CAUSE ANALYSIS

### Problem 1: Global Sense IDs (The Biggest Bug)
```
OLD (WRONG):
  জল,  sense_id=1  → 'অশ্রু'
  মাথা, sense_id=1  → 'শরীরের অঙ্গ'   ← SAME ID, DIFFERENT MEANING!
  তারা, sense_id=1  → 'নক্ষত্র'        ← SAME ID, DIFFERENT MEANING!

The model learned: sense_id=1 means EVERYTHING simultaneously.
Result: completely random predictions.

FIXED:
  জল_অশ্রু     → unique label
  মাথা_শরীর    → unique label   ← different labels!
  তারা_নক্ষত্র  → unique label   ← different labels!
```

### Problem 2: No Target Word Markers
```
OLD INPUT: "তার চোখে জল এলো"
  → Model sees all words equally, can't focus on জল
  → It might associate 'চোখ' or 'এলো' with the sense

FIXED INPUT: "তার চোখে [TGT] জল [/TGT] এলো"
  → [TGT] token draws BERT attention to জল
  → Model correctly focuses on the ambiguous word
```

### Problem 3: Wrong Inference Logic
```
OLD: predict_sense(sentence)    ← target_word ignored!
  → Predicts global sense IDs regardless of which word
  → 'জল' and 'মাথা' go through the SAME label space

FIXED: predict_sense(sentence, target_word)
  → Inserts [TGT] markers around target_word
  → Filters label space to ONLY that word's senses
  → Returns word-specific prediction
```

### Problem 4: No Input Validation
```
OLD: If target_word not in sentence → random prediction (silent failure)
FIXED: Returns error message immediately before any forward pass
```

### Problem 5: No Regularization → Overfitting
```
OLD: No dropout, no weight decay, no early stopping
     → 99% train accuracy, 42% val accuracy = pure memorization

FIXED:
  - Dropout=0.3 on classification head
  - Weight decay=0.01 in AdamW
  - Early stopping (patience=3)
  - Gradient clipping (max_norm=1.0)
  - Weighted loss for class imbalance
```

## ✅ WHY THE FIXED ARCHITECTURE WORKS

| Component | Old | Fixed | Why Better |
|-----------|-----|-------|------------|
| Label space | Global sense_ids | word_sense labels | No cross-word confusion |
| Input | Raw sentence | [TGT] marked sentence | BERT attends to right word |
| Inference | Ignores target_word | Filters to word's senses | Correct disambiguation |
| Regularization | None | Dropout + WD + ES | Generalizes to new sentences |
| Validation | Fake high accuracy | Word-specific accuracy | Honest evaluation |

## 📦 Step 1: Install & Import Libraries

In [2]:
# Install required libraries
# Kaggle has most pre-installed, but we pin transformers for stability
!pip install transformers>=4.36.0 --quiet
!pip install scikit-learn --quiet
!pip install openpyxl --quiet

# Disable the tokenizers parallelism warning (safe for notebooks)
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Libraries installed!')

✅ Libraries installed!


In [3]:
# ─────────────────────────────────────────────────────────────────
# CORE IMPORTS
# ─────────────────────────────────────────────────────────────────
import os
import re
import pickle
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Prevent tokenizer warnings

print('✅ All libraries imported successfully!')

✅ All libraries imported successfully!


## 🎲 Step 2: Reproducibility & Device Setup

In [4]:
def set_seed(seed: int = 42):
    """
    Fix all random seeds for full reproducibility.
    Without this, PyTorch initializes weights differently each run,
    so you get different results every time — bad for debugging.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('   ⚠️  No GPU found — training will be slow. Enable GPU in Kaggle settings.')

🖥️  Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB


## ⚙️ Step 3: Configuration (All Hyperparameters in One Place)

In [5]:
# ─────────────────────────────────────────────────────────────────
# CONFIGURATION CLASS
# All settings are centralized here so you can tune without
# hunting through the notebook.
# ─────────────────────────────────────────────────────────────────
class CFG:
    # ── Model ────────────────────────────────────────────────────
    MODEL_NAME    = 'sagorsarker/bangla-bert-base'
    MAX_LEN       = 128      # Max token length per sample

    # ── Training ─────────────────────────────────────────────────
    EPOCHS        = 15       # Max epochs (early stopping will cut this short)
    BATCH_SIZE    = 16       # Reduce to 8 if you get CUDA OOM
    LEARNING_RATE = 2e-5     # Standard for BERT fine-tuning
    WEIGHT_DECAY  = 0.01     # L2 regularization in AdamW
    WARMUP_RATIO  = 0.1      # 10% of steps used for LR warm-up
    MAX_GRAD_NORM = 1.0      # Gradient clipping prevents exploding gradients

    # ── Regularization ───────────────────────────────────────────
    DROPOUT_RATE  = 0.3      # Applied on classification head
    PATIENCE      = 4        # Early stopping: stop if no improvement for 4 epochs

    # ── Data ─────────────────────────────────────────────────────
    VAL_SIZE      = 0.15     # 15% of data held out for validation
    SEED          = 42
    MIN_SAMPLES   = 3        # Drop senses with fewer than this many samples

    # ── Paths ────────────────────────────────────────────────────
    # Adjust DATA_PATH to your Kaggle dataset path
    DATA_PATH     = '/kaggle/input/datasets/provapaul/bangla-wsd-dataset/dataset_updated.xlsx'
    MODEL_SAVE_PATH = '/kaggle/working/best_wsd_model'

    # ── [TGT] Marker Tokens ──────────────────────────────────────
    # CRITICAL: We do NOT add these as new tokens (that would resize embeddings).
    # Instead we use them as plain text — BanglaBERT will subword-tokenize them,
    # and the model learns their meaning purely from positional context.
    TGT_START = '[TGT]'
    TGT_END   = '[/TGT]'

print('✅ Configuration loaded!')
print(f'   Model: {CFG.MODEL_NAME}')
print(f'   Max length: {CFG.MAX_LEN}')
print(f'   Batch size: {CFG.BATCH_SIZE}')
print(f'   Epochs: {CFG.EPOCHS} (with early stopping, patience={CFG.PATIENCE})')

✅ Configuration loaded!
   Model: sagorsarker/bangla-bert-base
   Max length: 128
   Batch size: 16
   Epochs: 15 (with early stopping, patience=4)


## 📊 Step 4: Load & Validate Dataset

In [6]:
# ─────────────────────────────────────────────────────────────────
# DATASET LOADING
# Supports CSV, Excel (.xlsx), and TSV formats.
# Expected columns: target_word, sense_id, sense_gloss, text
# ─────────────────────────────────────────────────────────────────

def load_dataset(path: str) -> pd.DataFrame:
    """Load dataset from CSV, Excel, or TSV file."""
    ext = path.split('.')[-1].lower()
    if ext == 'csv':
        df = pd.read_csv(path)
    elif ext in ['xlsx', 'xls']:
        df = pd.read_excel(path)
    elif ext == 'tsv':
        df = pd.read_csv(path, sep='\t')
    else:
        raise ValueError(f'Unsupported file format: {ext}')
    return df

# ── Load ───────────────────────────────────────────────────────────
df = load_dataset(CFG.DATA_PATH)

print(f'📂 Loaded {len(df)} rows from {CFG.DATA_PATH}')
print(f'\n📋 Columns: {list(df.columns)}')
print(f'\n🔍 Sample rows:')
display(df.head(5))
print(f'\n📊 Basic stats:')
print(f'   Unique target words: {df["target_word"].nunique()}')
print(f'   Unique sense_ids:    {df["sense_id"].nunique()}')
print(f'   Unique glosses:      {df["sense_gloss"].nunique()}')

📂 Loaded 7409 rows from /kaggle/input/datasets/provapaul/bangla-wsd-dataset/dataset_updated.xlsx

📋 Columns: ['word_id', 'target_word', 'sense_id', 'sense_gloss', 'text']

🔍 Sample rows:


,word_id,target_word,sense_id,sense_gloss,text
0,1,জল,1,অত্যাধিক পরিশ্রম,মাঠে সারাদিন রক্ত জল করা পরিশ্রম করেও কৃষক ন্য...
1,1,জল,1,অত্যাধিক পরিশ্রম,বছরের পর বছর রক্ত জল করা পরিশ্রম করে তিনি এই ক...
2,1,জল,1,অত্যাধিক পরিশ্রম,মা সন্তানের লেখাপড়ার জন্য সারাজীবন রক্ত জল কর...
3,1,জল,1,অত্যাধিক পরিশ্রম,অলিম্পিকে পদক পেতে হলে বছরের পর বছর রক্ত জল কর...
4,1,জল,1,অত্যাধিক পরিশ্রম,"শ্রমিকরা রক্ত জল করা পরিশ্রম করেন, কিন্তু সেই ..."



📊 Basic stats:
   Unique target words: 181
   Unique sense_ids:    4
   Unique glosses:      637


## 🧹 Step 5: Full Preprocessing Pipeline

In [7]:
# ─────────────────────────────────────────────────────────────────
# PREPROCESSING PIPELINE
# Each step removes broken or misleading samples that would
# confuse the model during training.
# ─────────────────────────────────────────────────────────────────

print('🔧 Starting preprocessing pipeline...')
print(f'   Initial size: {len(df)} rows')

# ── Step 5.1: Drop NaN values ─────────────────────────────────────
before = len(df)
df = df.dropna(subset=['target_word', 'sense_id', 'sense_gloss', 'text'])
print(f'   After NaN removal: {len(df)} rows (dropped {before - len(df)})')

# ── Step 5.2: Strip whitespace and normalize types ────────────────
df['target_word'] = df['target_word'].astype(str).str.strip()
df['sense_id']    = df['sense_id'].astype(str).str.strip()
df['sense_gloss'] = df['sense_gloss'].astype(str).str.strip()
df['text']        = df['text'].astype(str).str.strip()

# Normalize multiple spaces
df['text'] = df['text'].apply(lambda x: re.sub(r'\s+', ' ', x))
print('   ✅ Whitespace normalized')

# ── Step 5.3: Remove empty text rows ─────────────────────────────
before = len(df)
df = df[df['text'].str.len() > 5]  # At least 5 chars
print(f'   After empty text removal: {len(df)} rows (dropped {before - len(df)})')

# ── Step 5.4: CRITICAL CHECK — target_word must exist in text ──────
# If the sentence doesn't contain the target word,
# the [TGT] marker can't be inserted → broken sample → remove it.
before = len(df)
df = df[df.apply(lambda row: row['target_word'] in row['text'], axis=1)]
dropped_no_word = before - len(df)
print(f'   After target-word-in-text check: {len(df)} rows (dropped {dropped_no_word})')
if dropped_no_word > 0:
    print(f'   ⚠️  {dropped_no_word} rows had target_word not found in sentence — REMOVED')

# ── Step 5.5: Remove duplicate rows ───────────────────────────────
# IMPORTANT: duplicates cause data leakage (same sentence in train AND val)
before = len(df)
df = df.drop_duplicates(subset=['text', 'sense_id', 'target_word'])
print(f'   After duplicate removal: {len(df)} rows (dropped {before - len(df)})')

# ── Step 5.6: Reset index ─────────────────────────────────────────
df = df.reset_index(drop=True)
print(f'\n✅ Preprocessing complete! Final size: {len(df)} rows')

🔧 Starting preprocessing pipeline...
   Initial size: 7409 rows
   After NaN removal: 7409 rows (dropped 0)
   ✅ Whitespace normalized
   After empty text removal: 7409 rows (dropped 0)
   After target-word-in-text check: 7390 rows (dropped 19)
   ⚠️  19 rows had target_word not found in sentence — REMOVED
   After duplicate removal: 7389 rows (dropped 1)

✅ Preprocessing complete! Final size: 7389 rows


## 🏷️ Step 6: Build Word-Specific Labels (THE KEY FIX)

In [8]:
# ─────────────────────────────────────────────────────────────────
# WORD-SPECIFIC LABEL CREATION
#
# WHY THIS IS THE MOST IMPORTANT FIX:
#
# OLD approach:
#   জল,  sense_id=1  → label=1   (means 'অশ্রু')
#   মাথা, sense_id=1  → label=1   (means 'শরীরের অঙ্গ')  ← COLLISION!
#   তারা, sense_id=1  → label=1   (means 'নক্ষত্র')       ← COLLISION!
#
#   The model tries to learn: label 1 = অশ্রু AND শরীরের অঙ্গ AND নক্ষত্র
#   This is mathematically impossible. The gradients cancel each other.
#   Result: random predictions, stuck at ~25% accuracy.
#
# NEW approach (word_sensegloss as label):
#   জল_অশ্রু          → unique label  (only জল's 'অশ্রু' sense)
#   মাথা_শরীরের_অঙ্গ   → unique label  (only মাথা's 'শরীরের অঙ্গ' sense)
#   তারা_নক্ষত্র       → unique label  (only তারা's 'নক্ষত্র' sense)
#
#   Each label is UNAMBIGUOUS. The model can learn correctly.
# ─────────────────────────────────────────────────────────────────

def build_word_specific_label(row):
    """
    Create a unique label by combining target_word + sense_gloss.
    Example: জল + অশ্রু → 'জল_অশ্রু'

    We use sense_gloss (human-readable meaning) instead of sense_id
    because sense_id=1 means different things for different words.
    Gloss is specific and readable.
    """
    # Normalize the gloss: strip, replace spaces with underscore
    gloss_clean = row['sense_gloss'].strip().replace(' ', '_')
    return f"{row['target_word']}_{gloss_clean}"

# Create the word-specific label column
df['word_sense_label'] = df.apply(build_word_specific_label, axis=1)

# ── Remove rare senses (< MIN_SAMPLES) ────────────────────────────
# Senses with very few examples can't be stratified-split properly
# and the model can't learn from 1-2 examples.
label_counts = df['word_sense_label'].value_counts()
valid_labels = label_counts[label_counts >= CFG.MIN_SAMPLES].index
before = len(df)
df = df[df['word_sense_label'].isin(valid_labels)].reset_index(drop=True)
print(f'Removed {before - len(df)} rows with rare senses (< {CFG.MIN_SAMPLES} samples)')

# ── Build label encoder ───────────────────────────────────────────
# LabelEncoder converts string labels → integers for PyTorch
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['word_sense_label'])
NUM_CLASSES = len(label_encoder.classes_)

# ── Build label → sense info mapping ─────────────────────────────
# This maps integer labels back to human-readable sense info
label_to_sense = {}
for _, row in df.drop_duplicates('word_sense_label').iterrows():
    lbl = label_encoder.transform([row['word_sense_label']])[0]
    label_to_sense[lbl] = {
        'word_sense': row['word_sense_label'],
        'target_word': row['target_word'],
        'sense_id': row['sense_id'],
        'gloss': row['sense_gloss'],
    }

# ── Build word → label indices mapping ───────────────────────────
# CRITICAL for inference: when predicting for 'জল', we only
# consider labels belonging to 'জল', not 'মাথা' or 'তারা'
word_to_label_indices = {}
for word, group in df.groupby('target_word'):
    word_to_label_indices[word] = sorted(group['label'].unique().tolist())

print(f'\n✅ Word-Specific Label Summary:')
print(f'   Total unique labels (word+sense): {NUM_CLASSES}')
print(f'   Total target words: {len(word_to_label_indices)}')
print(f'   Dataset size after filtering: {len(df)}')
print(f'\n📋 Sample labels:')
for lbl, info in list(label_to_sense.items())[:8]:
    print(f'   [{lbl:3d}] {info["word_sense"]}')

print(f'\n🔑 Word → Senses mapping (first 5 words):')
for word, indices in list(word_to_label_indices.items())[:5]:
    senses = [label_to_sense[i]['gloss'] for i in indices]
    print(f'   {word}: {senses}')

Removed 8 rows with rare senses (< 3 samples)

✅ Word-Specific Label Summary:
   Total unique labels (word+sense): 657
   Total target words: 179
   Dataset size after filtering: 7381

📋 Sample labels:
   [206] জল_অত্যাধিক_পরিশ্রম
   [208] জল_বৃষ্টি
   [207] জল_অশ্রু
   [ 17] আগ_আগুন
   [ 18] আগ_ওহে
   [ 16] আগ_অগ্রভাগ
   [408] বর্ণ_রঙ
   [409] বর্ণ_হিন্দুসমাজের_বিভাগ

🔑 Word → Senses mapping (first 5 words):
   অঙ্গ: ['উপাদান', 'বিভাগ', 'শরীরের অংশ', 'সহযোগী অংশ']
   অজ: ['অজাত (যে জন্মগ্রহণ করেনি)', 'নিতান্ত', 'মদ্দা ছাগল']
   অন্তর: ['ব্যবধান', 'মতান্তর', 'হৃদয়ের অন্তর্গত']
   অর্থ: ['অভিপ্রায়', 'ধন-দৌলত', 'বিষয়']
   আকা: ['বড় উনান', 'মালিক', 'সূর্য']


## 🏷️ Step 7: Insert [TGT] Markers into Sentences

In [9]:
# ─────────────────────────────────────────────────────────────────
# [TGT] MARKER INSERTION
#
# WHY WE NEED [TGT] MARKERS:
#
# BanglaBERT processes the ENTIRE sentence with self-attention.
# Without markers, every token gets equal initial weight.
# The model must "guess" which word needs disambiguation.
#
# With [TGT] markers:
#   "তার চোখে [TGT] জল [/TGT] এলো"
#   → [TGT] and [/TGT] create strong positional signals
#   → BERT's attention heads learn to focus on tokens
#      surrounded by these markers
#   → The [CLS] pooled representation becomes strongly
#      influenced by the marked word's local context
#
# NOTE: We do NOT use tokenizer.add_tokens() or resize_embeddings().
# [TGT] and [/TGT] are treated as plain Bangla text.
# BanglaBERT will subword-tokenize them, but the model still learns
# their boundary-marking role from position + context.
# ─────────────────────────────────────────────────────────────────

def insert_tgt_markers(sentence: str, target_word: str) -> str | None:
    """
    Insert [TGT] ... [/TGT] markers around the FIRST occurrence of
    target_word in sentence.

    Returns:
        Marked sentence string, or None if target_word not found.

    Examples:
        insert_tgt_markers('তার চোখে জল এলো', 'জল')
        → 'তার চোখে [TGT] জল [/TGT] এলো'

        insert_tgt_markers('মাথা ব্যথা করছে', 'মাথা')
        → '[TGT] মাথা [/TGT] ব্যথা করছে'
    """
    if target_word not in sentence:
        return None  # Signal to caller that this sample is broken

    # Use word boundary regex to match the exact word
    # (avoids partial matches like 'জল' inside 'জলপ্রপাত')
    # For Bangla, we use simple string replacement on first occurrence
    marked = sentence.replace(
        target_word,
        f'{CFG.TGT_START} {target_word} {CFG.TGT_END}',
        1  # Replace only FIRST occurrence
    )
    return marked

# Apply to entire dataset
df['marked_text'] = df.apply(
    lambda row: insert_tgt_markers(row['text'], row['target_word']),
    axis=1
)

# Validate: drop rows where marker insertion failed
before = len(df)
df = df.dropna(subset=['marked_text']).reset_index(drop=True)
print(f'Removed {before - len(df)} rows where TGT insertion failed')

# Show examples
print('\n✅ [TGT] Marker Examples:')
print('=' * 65)
for _, row in df.head(5).iterrows():
    print(f'  Original: {row["text"]}')
    print(f'  Marked:   {row["marked_text"]}')
    print(f'  Label:    {row["word_sense_label"]}')
    print('-' * 65)

Removed 0 rows where TGT insertion failed

✅ [TGT] Marker Examples:
  Original: মাঠে সারাদিন রক্ত জল করা পরিশ্রম করেও কৃষক ন্যায্য মূল্য থেকে বঞ্চিত হন।
  Marked:   মাঠে সারাদিন রক্ত [TGT] জল [/TGT] করা পরিশ্রম করেও কৃষক ন্যায্য মূল্য থেকে বঞ্চিত হন।
  Label:    জল_অত্যাধিক_পরিশ্রম
-----------------------------------------------------------------
  Original: বছরের পর বছর রক্ত জল করা পরিশ্রম করে তিনি এই কারখানাটি দাঁড় করিয়েছেন।
  Marked:   বছরের পর বছর রক্ত [TGT] জল [/TGT] করা পরিশ্রম করে তিনি এই কারখানাটি দাঁড় করিয়েছেন।
  Label:    জল_অত্যাধিক_পরিশ্রম
-----------------------------------------------------------------
  Original: মা সন্তানের লেখাপড়ার জন্য সারাজীবন রক্ত জল করা পরিশ্রম করে গেছেন।
  Marked:   মা সন্তানের লেখাপড়ার জন্য সারাজীবন রক্ত [TGT] জল [/TGT] করা পরিশ্রম করে গেছেন।
  Label:    জল_অত্যাধিক_পরিশ্রম
-----------------------------------------------------------------
  Original: অলিম্পিকে পদক পেতে হলে বছরের পর বছর রক্ত জল করা পরিশ্রম করতে হয়।
  Marked:   অলিম্পিকে পদক

## ✂️ Step 8: Stratified Train/Val Split

In [10]:
# ─────────────────────────────────────────────────────────────────
# STRATIFIED SPLIT
#
# WHY STRATIFIED:
# Regular random split might put ALL examples of a rare sense
# into training, leaving validation with 0 examples of that sense.
# The model then gets 99% accuracy (never wrong on rare senses)
# but fails completely at inference time.
#
# Stratified split GUARANTEES each sense appears in both splits
# at the same proportion as the full dataset.
# ─────────────────────────────────────────────────────────────────

train_df, val_df = train_test_split(
    df,
    test_size=CFG.VAL_SIZE,
    random_state=CFG.SEED,
    stratify=df['label'],     # ← THIS is the key: stratify by label
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'✅ Stratified Split:')
print(f'   Train: {len(train_df)} samples ({len(train_df)/len(df)*100:.1f}%)')
print(f'   Val:   {len(val_df)} samples ({len(val_df)/len(df)*100:.1f}%)')
print(f'   Labels in train: {train_df["label"].nunique()}')
print(f'   Labels in val:   {val_df["label"].nunique()}')

# Sanity check: both splits should have same number of unique labels
assert train_df['label'].nunique() == val_df['label'].nunique(), \
    'ERROR: val set is missing some labels! Increase data or reduce MIN_SAMPLES.'

✅ Stratified Split:
   Train: 6273 samples (85.0%)
   Val:   1108 samples (15.0%)
   Labels in train: 657
   Labels in val:   657


## 🔤 Step 9: Load BanglaBERT Tokenizer

In [11]:
# ─────────────────────────────────────────────────────────────────
# TOKENIZER LOADING
#
# IMPORTANT NOTES:
# 1. We do NOT call tokenizer.add_tokens([TGT_START, TGT_END])
#    because that would add new vocabulary entries requiring
#    embedding resize — unstable with pre-trained weights.
#
# 2. [TGT] and [/TGT] are plain text strings that BanglaBERT
#    will tokenize as unknown subwords. The model still learns
#    their role from positional context and gradient descent.
#
# 3. use_fast=True uses the Rust-based tokenizer (faster, safer).
# ─────────────────────────────────────────────────────────────────

print(f'📥 Loading tokenizer from: {CFG.MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME, use_fast=True)

print(f'✅ Tokenizer loaded!')
print(f'   Vocab size: {tokenizer.vocab_size}')
print(f'   Max model length: {tokenizer.model_max_length}')

# Test tokenization on a marked sentence
test_sentence = 'তার চোখে [TGT] জল [/TGT] এলো'
tokens = tokenizer.tokenize(test_sentence)
print(f'\n🔬 Tokenization test:')
print(f'   Input:  "{test_sentence}"')
print(f'   Tokens: {tokens}')

📥 Loading tokenizer from: sagorsarker/bangla-bert-base


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded!
   Vocab size: 101975
   Max model length: 1000000000000000019884624838656

🔬 Tokenization test:
   Input:  "তার চোখে [TGT] জল [/TGT] এলো"
   Tokens: ['তার', 'চোখে', '[', '[UNK]', ']', 'জল', '[', '/', '[UNK]', ']', 'এলো']


## 📦 Step 10: PyTorch Dataset & DataLoader

In [12]:
# ─────────────────────────────────────────────────────────────────
# PYTORCH DATASET
#
# The dataset takes marked_text (with [TGT] markers) and
# returns tokenized tensors for BanglaBERT.
#
# We use ONLY marked_text as input — no 'word: sentence' format,
# no extra context. The markers already tell the model
# which word to focus on.
# ─────────────────────────────────────────────────────────────────

class WSDDataset(Dataset):
    """
    PyTorch Dataset for Bangla WSD.
    Input: marked sentences with [TGT] ... [/TGT] markers.
    Output: input_ids, attention_mask, labels for BERT.
    """
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_len: int):
        self.data      = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Use the marked sentence (with [TGT] markers)
        text = row['marked_text']
        label = int(row['label'])

        # Tokenize: BanglaBERT expects:
        #   [CLS] token1 token2 ... [SEP]
        # max_length + truncation ensures consistent tensor sizes
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',    # Pad shorter sequences
            truncation=True,          # Truncate longer sequences
            return_tensors='pt',      # Return PyTorch tensors
            return_attention_mask=True,
        )

        return {
            'input_ids':      encoding['input_ids'].squeeze(0),      # (max_len,)
            'attention_mask': encoding['attention_mask'].squeeze(0),  # (max_len,)
            'labels':         torch.tensor(label, dtype=torch.long),  # scalar
        }


# ── Create Datasets ───────────────────────────────────────────────
train_dataset = WSDDataset(train_df, tokenizer, CFG.MAX_LEN)
val_dataset   = WSDDataset(val_df,   tokenizer, CFG.MAX_LEN)

# ── Create DataLoaders ────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,       # Shuffle training data each epoch
    num_workers=2,      # Parallel data loading
    pin_memory=True,    # Speed up GPU transfer
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.BATCH_SIZE * 2,  # Larger batch for validation (no gradients)
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f'✅ Datasets created!')
print(f'   Train batches: {len(train_loader)}')
print(f'   Val batches:   {len(val_loader)}')

# Test a batch
sample_batch = next(iter(train_loader))
print(f'\n🔬 Sample batch shapes:')
print(f'   input_ids:      {sample_batch["input_ids"].shape}')
print(f'   attention_mask: {sample_batch["attention_mask"].shape}')
print(f'   labels:         {sample_batch["labels"].shape}')

✅ Datasets created!
   Train batches: 393
   Val batches:   35

🔬 Sample batch shapes:
   input_ids:      torch.Size([16, 128])
   attention_mask: torch.Size([16, 128])
   labels:         torch.Size([16])


## 🧠 Step 11: BanglaBERT WSD Model Architecture

In [13]:
# ─────────────────────────────────────────────────────────────────
# MODEL ARCHITECTURE
#
# Architecture:
#   Input Sentence (with [TGT] markers)
#       ↓
#   BanglaBERT Encoder (12 transformer layers)
#       ↓
#   [CLS] token representation (768-dim)
#       ↓
#   LayerNorm → Dropout(0.3) → Linear(768, 768) → GELU
#       ↓
#   Dropout(0.3) → Linear(768, NUM_CLASSES)
#       ↓
#   Word-Specific Logits (NUM_CLASSES outputs)
#
# WHY [CLS] POOLING:
# BanglaBERT's [CLS] token attends to ALL other tokens
# through self-attention. After fine-tuning, it learns to
# concentrate on the [TGT]-marked word for classification.
#
# WHY TWO DROPOUT LAYERS:
# More aggressive regularization → better generalization.
# The model is 110M parameters; we only have a few thousand
# training examples → high overfitting risk.
# ─────────────────────────────────────────────────────────────────

class BanglaBERTWSD(nn.Module):
    """
    BanglaBERT-based Word Sense Disambiguation model.

    Uses [CLS] representation from BanglaBERT + classification head.
    """
    def __init__(self, model_name: str, num_classes: int, dropout_rate: float = 0.3):
        super().__init__()
        self.num_classes = num_classes

        # ── BanglaBERT Encoder ────────────────────────────────────
        # AutoModel gives us BERT without a pre-trained head.
        # We add our own classification head on top.
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 768 for BERT-base

        # ── Classification Head ───────────────────────────────────
        # LayerNorm: stabilizes training, speeds convergence
        # Two-layer MLP: captures non-linear relationships
        # GELU: smooth activation (better than ReLU for transformers)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_size, num_classes),
        )

        # Initialize the classification head weights properly
        self._init_weights()

    def _init_weights(self):
        """Initialize classification head with small normal weights."""
        for module in self.classifier.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, input_ids, attention_mask):
        """
        Forward pass.

        Args:
            input_ids:      (batch, seq_len) token IDs
            attention_mask: (batch, seq_len) 1=real token, 0=padding

        Returns:
            logits: (batch, num_classes) raw scores
        """
        # BanglaBERT output: last_hidden_state, pooler_output
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # [CLS] token is at position 0
        # Shape: (batch, hidden_size) = (batch, 768)
        cls_repr = outputs.last_hidden_state[:, 0, :]

        # Classification head: (batch, 768) → (batch, num_classes)
        logits = self.classifier(cls_repr)
        return logits


# ── Instantiate model ─────────────────────────────────────────────
print(f'📥 Loading BanglaBERT from: {CFG.MODEL_NAME}')
model = BanglaBERTWSD(
    model_name=CFG.MODEL_NAME,
    num_classes=NUM_CLASSES,
    dropout_rate=CFG.DROPOUT_RATE,
)
model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model loaded!')
print(f'   Total parameters: {total_params/1e6:.1f}M')
print(f'   Trainable parameters: {trainable_params/1e6:.1f}M')
print(f'   Classification head outputs: {NUM_CLASSES} classes')

📥 Loading BanglaBERT from: sagorsarker/bangla-bert-base


model.safetensors:   0%|          | 0.00/660M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded!
   Total parameters: 165.5M
   Trainable parameters: 165.5M
   Classification head outputs: 657 classes


## ⚖️ Step 12: Class Weights, Optimizer & Scheduler

In [14]:
# ─────────────────────────────────────────────────────────────────
# WEIGHTED LOSS FOR CLASS IMBALANCE
#
# Problem: If 'জল_পানি' has 200 samples but 'জল_অশ্রু' has 20,
# the model learns to always predict 'জল_পানি' and gets
# 200/220 = 91% accuracy by ignoring the rare class.
#
# Solution: Weight rare classes higher so the model is
# penalized MORE for getting them wrong.
# class_weight = total_samples / (n_classes * class_count)
# ─────────────────────────────────────────────────────────────────

# Compute class weights from training data only (not val!)
train_labels = train_df['label'].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_labels,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

# Weighted CrossEntropyLoss
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(f'⚖️  Class weights computed (top-5 highest weighted):')
sorted_weights = sorted(enumerate(class_weights), key=lambda x: -x[1])
for lbl_idx, w in sorted_weights[:5]:
    sense = label_to_sense.get(lbl_idx, {}).get('word_sense', '?')
    count = (train_df['label'] == lbl_idx).sum()
    print(f'   [{lbl_idx:3d}] {sense:30s} weight={w:.3f} (n={count})')

# ── Optimizer ─────────────────────────────────────────────────────
# AdamW: Adam with proper weight decay (not applied to biases/LayerNorm).
# We use different learning rates for BERT vs classifier head.
# This is called 'discriminative fine-tuning' — classifier learns faster.
optimizer = AdamW(
    [
        {'params': model.bert.parameters(), 'lr': CFG.LEARNING_RATE},
        {'params': model.classifier.parameters(), 'lr': CFG.LEARNING_RATE * 10},
    ],
    weight_decay=CFG.WEIGHT_DECAY,
)

# ── Learning Rate Scheduler ───────────────────────────────────────
# Linear warmup: LR gradually increases for first 10% of steps.
# This prevents catastrophic forgetting of pre-trained weights.
# After warmup, LR linearly decays to 0 by end of training.
total_steps  = len(train_loader) * CFG.EPOCHS
warmup_steps = int(total_steps * CFG.WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'\n✅ Optimizer & Scheduler ready:')
print(f'   BERT LR: {CFG.LEARNING_RATE}')
print(f'   Classifier LR: {CFG.LEARNING_RATE * 10}')
print(f'   Total steps: {total_steps}')
print(f'   Warmup steps: {warmup_steps}')

⚖️  Class weights computed (top-5 highest weighted):
   [562] রাজা_শাসক                      weight=1.591 (n=6)
   [176] চাবি_সমাধানের_উপায়            weight=1.364 (n=7)
   [511] মাধ্যম_যোগাযোগের_উপায়         weight=1.364 (n=7)
   [  1] অঙ্গ_বিভাগ                     weight=1.193 (n=8)
   [  2] অঙ্গ_শরীরের_অংশ                weight=1.193 (n=8)

✅ Optimizer & Scheduler ready:
   BERT LR: 2e-05
   Classifier LR: 0.0002
   Total steps: 5895
   Warmup steps: 589


## 🏋️ Step 13: Training Loop with Early Stopping

In [15]:
# ─────────────────────────────────────────────────────────────────
# TRAINING & VALIDATION FUNCTIONS
# ─────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    """
    Run one full pass over the training data.
    Returns: average loss, accuracy.
    """
    model.train()  # Enable dropout and batch norm updates
    total_loss = 0
    correct    = 0
    total      = 0

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()  # Clear gradients from previous batch

        # Forward pass
        logits = model(input_ids, attention_mask)  # (batch, num_classes)

        # Compute loss
        loss = criterion(logits, labels)

        # Backward pass: compute gradients
        loss.backward()

        # Gradient clipping: prevents exploding gradients in BERT
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.MAX_GRAD_NORM)

        # Update weights
        optimizer.step()
        scheduler.step()  # Update learning rate

        # Track metrics
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)  # Predicted class
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return total_loss / len(loader), correct / total


@torch.no_grad()  # No gradients needed during validation → saves memory
def evaluate(model, loader, criterion, device):
    """
    Evaluate model on a DataLoader.
    Returns: average loss, accuracy, all predictions, all true labels.
    """
    model.eval()  # Disable dropout (deterministic predictions)
    total_loss = 0
    all_preds  = []
    all_labels = []

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)

        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc, all_preds, all_labels


print('✅ Training functions defined!')
print('   Ready to start training.')

✅ Training functions defined!
   Ready to start training.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# MAIN TRAINING LOOP WITH EARLY STOPPING
#
# Early stopping: if val_accuracy doesn't improve for PATIENCE
# consecutive epochs, stop training and load the best checkpoint.
# This prevents overfitting from running too many epochs.
# ─────────────────────────────────────────────────────────────────

# Training history for plotting
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
}

best_val_acc  = 0.0
best_epoch    = 0
patience_ctr  = 0
best_model_state = None

print('🚀 Starting training...')
print('=' * 70)

for epoch in range(1, CFG.EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────────
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, DEVICE
    )

    # ── Validate ───────────────────────────────────────────────────
    val_loss, val_acc, val_preds, val_labels = evaluate(
        model, val_loader, criterion, DEVICE
    )

    # ── Record history ─────────────────────────────────────────────
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # ── Check for improvement ──────────────────────────────────────
    improved = val_acc > best_val_acc
    if improved:
        best_val_acc  = val_acc
        best_epoch    = epoch
        patience_ctr  = 0
        # Save a deep copy of current model weights
        best_model_state = {
            k: v.clone() for k, v in model.state_dict().items()
        }
        marker = '  ← 🏆 BEST'
    else:
        patience_ctr += 1
        marker = f'  (patience {patience_ctr}/{CFG.PATIENCE})'

    print(
        f'Epoch [{epoch:2d}/{CFG.EPOCHS}] '
        f'Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | '
        f'Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%'
        f'{marker}'
    )

    # ── Early stopping check ───────────────────────────────────────
    if patience_ctr >= CFG.PATIENCE:
        print(f'\n⏹️  Early stopping triggered at epoch {epoch}.')
        print(f'   No improvement for {CFG.PATIENCE} consecutive epochs.')
        break

print('\n' + '=' * 70)
print(f'✅ Training complete!')
print(f'   Best val accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}')

# ── Restore best model weights ─────────────────────────────────────
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f'   Best model weights restored from epoch {best_epoch}')

🚀 Starting training...
Epoch [ 1/15] Train Loss: 6.4136 Acc: 1.20% | Val Loss: 5.8426 Acc: 19.13%  ← 🏆 BEST
Epoch [ 2/15] Train Loss: 4.1359 Acc: 24.07% | Val Loss: 1.7663 Acc: 46.93%  ← 🏆 BEST
Epoch [ 3/15] Train Loss: 1.5130 Acc: 54.47% | Val Loss: 0.9085 Acc: 67.87%  ← 🏆 BEST
Epoch [ 4/15] Train Loss: 0.8036 Acc: 74.27% | Val Loss: 0.6550 Acc: 77.62%  ← 🏆 BEST
Epoch [ 5/15] Train Loss: 0.4524 Acc: 86.32% | Val Loss: 0.5115 Acc: 81.32%  ← 🏆 BEST
Epoch [ 6/15] Train Loss: 0.2479 Acc: 93.00% | Val Loss: 0.4645 Acc: 84.39%  ← 🏆 BEST
Epoch [ 7/15] Train Loss: 0.1305 Acc: 96.73% | Val Loss: 0.4351 Acc: 85.65%  ← 🏆 BEST
Epoch [ 8/15] Train Loss: 0.0727 Acc: 98.17% | Val Loss: 0.4568 Acc: 86.55%  ← 🏆 BEST
Epoch [ 9/15] Train Loss: 0.0388 Acc: 99.12% | Val Loss: 0.4212 Acc: 86.46%  (patience 1/4)
Epoch [10/15] Train Loss: 0.0207 Acc: 99.59% | Val Loss: 0.4175 Acc: 87.09%  ← 🏆 BEST
Epoch [11/15] Train Loss: 0.0105 Acc: 99.84% | Val Loss: 0.4558 Acc: 86.64%  (patience 1/4)


## 📈 Step 14: Training Curves & Overfitting Diagnosis

In [ ]:
# ─────────────────────────────────────────────────────────────────
# TRAINING CURVES
#
# HOW TO READ THESE PLOTS:
#
# HEALTHY training:
#   - Train and val loss both decrease together
#   - Train and val accuracy both increase together
#   - Small gap between train/val curves
#
# OVERFITTING (old model's problem):
#   - Train loss → 0, val loss stays high or increases
#   - Train acc → 99%, val acc stays low
#   - Large gap between train/val curves
# ─────────────────────────────────────────────────────────────────

epochs_ran = len(history['train_loss'])
x = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BanglaBERT WSD Training Curves', fontsize=14, fontweight='bold')

# ── Loss ───────────────────────────────────────────────────────────
axes[0].plot(x, history['train_loss'], 'b-o', label='Train Loss', markersize=5)
axes[0].plot(x, history['val_loss'],   'r-o', label='Val Loss',   markersize=5)
axes[0].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_epoch})')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Accuracy ──────────────────────────────────────────────────────
axes[1].plot(x, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc', markersize=5)
axes[1].plot(x, [a*100 for a in history['val_acc']],   'r-o', label='Val Acc',   markersize=5)
axes[1].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_epoch})')
axes[1].axhline(y=best_val_acc*100, color='orange', linestyle=':', alpha=0.7, label=f'Best Val Acc ({best_val_acc*100:.1f}%)')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 Training curves saved to /kaggle/working/training_curves.png')

## 📊 Step 15: Full Evaluation — Accuracy, F1, Confusion Matrix

In [ ]:
# ─────────────────────────────────────────────────────────────────
# FULL EVALUATION ON VALIDATION SET
# ─────────────────────────────────────────────────────────────────

print('📊 Running full evaluation on validation set...')

_, _, val_preds, val_labels = evaluate(model, val_loader, criterion, DEVICE)

val_preds  = np.array(val_preds)
val_labels = np.array(val_labels)

# ── Overall Metrics ───────────────────────────────────────────────
accuracy = accuracy_score(val_labels, val_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    val_labels, val_preds, average='weighted', zero_division=0
)

print('\n' + '═' * 60)
print('  OVERALL VALIDATION METRICS')
print('═' * 60)
print(f'  Accuracy:  {accuracy*100:.2f}%')
print(f'  Precision: {precision*100:.2f}%')
print(f'  Recall:    {recall*100:.2f}%')
print(f'  F1 Score:  {f1*100:.2f}%')
print('═' * 60)

# ── Per-Class Classification Report ──────────────────────────────
# Map integer labels back to readable names
label_names = [
    label_to_sense.get(i, {}).get('word_sense', f'class_{i}')
    for i in range(NUM_CLASSES)
]

# Only include classes that appear in val set
present_classes = sorted(set(val_labels.tolist()))
present_names   = [label_names[i] for i in present_classes]

print('\n📋 Per-Class Classification Report:')
print(classification_report(
    val_labels, val_preds,
    labels=present_classes,
    target_names=present_names,
    zero_division=0,
))

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────
# Shows which senses are being confused with each other.
# Diagonal = correct predictions.
# Off-diagonal = confusions.

# For readability, only show if fewer than 30 classes
if NUM_CLASSES <= 30:
    cm = confusion_matrix(val_labels, val_preds, labels=present_classes)

    fig_size = max(8, NUM_CLASSES // 2)
    plt.figure(figsize=(fig_size, fig_size * 0.85))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=present_names,
        yticklabels=present_names,
    )
    plt.title('Confusion Matrix — BanglaBERT WSD', fontsize=13, pad=15)
    plt.ylabel('True Label', fontsize=11)
    plt.xlabel('Predicted Label', fontsize=11)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 Confusion matrix saved to /kaggle/working/confusion_matrix.png')
else:
    print(f'ℹ️  {NUM_CLASSES} classes — confusion matrix too large to display.')
    print('   Showing per-word accuracy instead:')

# ── Per-Word Accuracy ─────────────────────────────────────────────
# Shows how well the model does for each target word individually
print('\n📋 Per-Word Accuracy on Validation Set:')
print('-' * 50)

val_df_eval = val_df.copy()
val_df_eval['pred_label'] = val_preds
val_df_eval['correct']    = (val_df_eval['label'] == val_df_eval['pred_label'])

per_word_acc = val_df_eval.groupby('target_word')['correct'].mean().sort_values()
for word, acc in per_word_acc.items():
    bar = '█' * int(acc * 20)
    print(f'  {word:15s} [{bar:<20s}] {acc*100:5.1f}%')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🌡️ TEMPERATURE SCALING — Fix overconfident predictions
# ─────────────────────────────────────────────────────────────────

class CalibratedPredictor:
    def __init__(self, model, tokenizer, label_to_sense,
                 word_to_label_indices, temperature=1.5):
        self.model                 = model
        self.tokenizer             = tokenizer
        self.label_to_sense        = label_to_sense
        self.word_to_label_indices = word_to_label_indices
        self.temperature           = temperature

    def find_best_temperature(self, val_loader,
                               temps=[0.5, 1.0, 1.5, 2.0, 2.5, 3.0]):
        """Grid search: find temperature that minimises NLL on val set."""
        self.model.eval()
        all_logits, all_labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels         = batch['labels'].to(DEVICE)
                logits         = self.model(input_ids, attention_mask)
                all_logits.append(logits.cpu())
                all_labels.append(labels.cpu())

        all_logits = torch.cat(all_logits)   # (N, NUM_CLASSES)
        all_labels = torch.cat(all_labels)   # (N,)
        nll        = nn.CrossEntropyLoss()

        best_temp, best_loss = 1.0, float('inf')
        print("🌡️  Temperature calibration search:")
        print("-" * 40)
        for t in temps:
            loss = nll(all_logits / t, all_labels).item()
            marker = " ← best" if loss < best_loss else ""
            print(f"   T = {t:.1f}  →  NLL Loss: {loss:.4f}{marker}")
            if loss < best_loss:
                best_loss = loss
                best_temp = t

        self.temperature = best_temp
        print(f"\n✅ Best temperature set to: {best_temp}")
        if best_temp >= 2.0:
            print("⚠️  High temperature needed → model was severely overconfident.")
            print("   This is a valid thesis finding — note it in your analysis.")
        return best_temp

    def predict(self, sentence, target_word):
        """Predict with temperature-scaled (calibrated) probabilities."""
        # ── Validation ────────────────────────────────────────────
        if not sentence or not sentence.strip():
            return {"error": "Empty sentence."}
        if target_word not in sentence:
            return {"error": f"'{target_word}' not found in sentence."}
        if target_word not in self.word_to_label_indices:
            return {"error": f"'{target_word}' not in training vocabulary."}

        # ── Insert [TGT] markers ──────────────────────────────────
        marked = sentence.replace(target_word,
                                   f"[TGT] {target_word} [/TGT]", 1)

        # ── Tokenize ──────────────────────────────────────────────
        encoding = self.tokenizer(
            marked,
            return_tensors = 'pt',
            max_length     = CFG.MAX_LEN,
            padding        = 'max_length',
            truncation     = True,
        )

        # ── Forward pass ──────────────────────────────────────────
        self.model.eval()
        with torch.no_grad():
            logits = self.model(
                encoding['input_ids'].to(DEVICE),
                encoding['attention_mask'].to(DEVICE)
            ).squeeze(0)                          # (NUM_CLASSES,)

        # ── Temperature scaling + word-specific filtering ─────────
        valid_indices = self.word_to_label_indices[target_word]
        word_logits   = logits[valid_indices] / self.temperature
        probs         = torch.softmax(word_logits, dim=-1).cpu().numpy()
        sorted_idx    = np.argsort(probs)[::-1]

        top_label = valid_indices[sorted_idx[0]]
        top_prob  = float(probs[sorted_idx[0]])
        top_sense = self.label_to_sense[top_label]['word_sense']
        top_gloss = self.label_to_sense[top_label]['gloss']

        # ── Confidence tag ────────────────────────────────────────
        if top_prob >= 0.80:
            conf_tag = "🟢 HIGH"
        elif top_prob >= 0.55:
            conf_tag = "🟡 MEDIUM"
        else:
            conf_tag = "🔴 LOW (ambiguous)"

        # ── Pretty print ──────────────────────────────────────────
        print("═" * 65)
        print(f"  📝 Sentence:    {sentence}")
        print(f"  🏷️  Marked:      {marked}")
        print(f"  🎯 Target word: {target_word}")
        print(f"  ✅ Predicted:   {top_sense}  (gloss: \"{top_gloss}\")")
        print(f"  📈 Confidence:  {top_prob*100:.2f}%  {conf_tag}")
        print(f"\n  🏆 Top Candidates:")
        for rank, i in enumerate(sorted_idx[:4]):
            lbl    = valid_indices[i]
            prefix = "  👑" if rank == 0 else f"    {rank+1}."
            ws     = self.label_to_sense[lbl]['word_sense']
            print(f"  {prefix} {ws:<40} → {probs[i]*100:.2f}%")
        print("═" * 65)

        return {
            "predicted_sense" : top_sense,
            "predicted_gloss" : top_gloss,
            "confidence"      : top_prob,
            "conf_level"      : conf_tag,
            "all_probs"       : {
                self.label_to_sense[valid_indices[i]]['word_sense']: float(probs[i])
                for i in sorted_idx
            },
        }


# ── Initialise & calibrate ────────────────────────────────────────
calibrated = CalibratedPredictor(
    model                 = model,
    tokenizer             = tokenizer,
    label_to_sense        = label_to_sense,
    word_to_label_indices = word_to_label_indices,
)
print("Finding best temperature on validation set...\n")
calibrated.find_best_temperature(val_loader)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🔴 HARD CASES TEST — Temporal & Metaphorical ambiguity
# ─────────────────────────────────────────────────────────────────

hard_cases = [
    # Temporal ambiguity — "কাল" সবচেয়ে কঠিন
    ("কাল সকালে আমরা রওনা দেব।",                         "কাল"),  # future
    ("কাল রাতে ভয়ংকর ঝড় হয়েছিল।",                      "কাল"),  # past
    ("কাল ঠান্ডার রাতে আগুন জ্বেলে উষ্ণ থাকা হলো।",      "কাল"),  # ভুল হয়েছিল

    # Metaphorical sense
    ("দেশের সুর নিশ্চিত করতে সীমান্তে সেনাবাহিনী মোতায়েন।", "সুর"),
    ("তার কণ্ঠে একটা আলাদা সুর ছিল।",                    "সুর"),

    # Compound-word boundary
    ("এই নথির একটি কপি আমাকে দাও।",                      "কপি"),
    ("ফুলকপি রান্না করলে সুস্বাদু হয়।",                   "কপি"),
]

print("🔴 HARD CASES — Calibrated Predictions")
print("=" * 65 + "\n")

results_summary = []
for sentence, word in hard_cases:
    result = calibrated.predict(sentence, word)
    if "error" in result:
        print(f"  ❌ Skipped '{word}': {result['error']}\n")
        continue

    level = result["conf_level"]
    if level.startswith("🔴"):
        print(f"  ⚠️  Model uncertain for '{word}' → good thesis finding!\n")
    elif level.startswith("🟡"):
        print(f"  ℹ️  Moderate confidence for '{word}'\n")
    else:
        print()

    results_summary.append({
        "sentence" : sentence,
        "word"     : word,
        "predicted": result["predicted_sense"],
        "conf"     : f"{result['confidence']*100:.1f}%",
        "level"    : level,
    })

# ── Summary table ─────────────────────────────────────────────────
print("\n📊 HARD CASES SUMMARY")
print("-" * 65)
print(f"  {'Word':<8} {'Confidence':<12} {'Level':<20} {'Predicted Sense'}")
print("-" * 65)
for r in results_summary:
    print(f"  {r['word']:<8} {r['conf']:<12} {r['level']:<20} {r['predicted']}")
print("-" * 65)

low_conf  = sum(1 for r in results_summary if r['level'].startswith("🔴"))
mid_conf  = sum(1 for r in results_summary if r['level'].startswith("🟡"))
high_conf = sum(1 for r in results_summary if r['level'].startswith("🟢"))
print(f"\n  🟢 HIGH confidence   : {high_conf}/{len(results_summary)}")
print(f"  🟡 MEDIUM confidence : {mid_conf}/{len(results_summary)}")
print(f"  🔴 LOW confidence    : {low_conf}/{len(results_summary)}")
print("\n  → LOW/MEDIUM cases = where model struggles = thesis error analysis section")

## 🔍 Step 16: Inference Pipeline — Correct Design

In [ ]:
# ─────────────────────────────────────────────────────────────────
# INFERENCE PIPELINE
#
# CRITICAL DIFFERENCE FROM OLD PIPELINE:
#
# OLD (broken):
#   predict(sentence) → runs through all 4 global sense_ids
#   → completely ignores target_word
#   → can predict 'অশ্রু' for a মাথা sentence
#
# NEW (correct):
#   predict(sentence, target_word):
#     1. Validate: target_word must be in sentence
#     2. Insert [TGT] markers around target_word
#     3. Tokenize the marked sentence
#     4. Run through BanglaBERT
#     5. Get logits for ALL classes
#     6. FILTER logits to ONLY this word's class indices
#     7. Apply softmax to filtered logits
#     8. Return word-specific prediction + confidence
# ─────────────────────────────────────────────────────────────────

@torch.no_grad()
def predict_sense(
    sentence: str,
    target_word: str,
    model: nn.Module,
    tokenizer,
    label_to_sense: dict,
    word_to_label_indices: dict,
    device: torch.device,
    max_len: int = CFG.MAX_LEN,
) -> dict:
    """
    Predict the sense of target_word in sentence.

    Args:
        sentence:    Input Bangla sentence
        target_word: The ambiguous word to disambiguate

    Returns:
        dict with keys:
          - success: bool
          - error: str (if success=False)
          - predicted_label: int
          - predicted_gloss: str
          - predicted_sense_id: str
          - confidence: float (0-1)
          - top_candidates: list of (label, gloss, probability)
          - marked_sentence: str
    """
    # ── Step 1: Input Validation ───────────────────────────────────
    if not sentence or not sentence.strip():
        return {'success': False, 'error': 'Empty sentence provided.'}

    if target_word not in sentence:
        return {
            'success': False,
            'error': f'Target word "{target_word}" not found in sentence. '
                     f'Cannot insert [TGT] markers. Refusing to predict.'
        }

    if target_word not in word_to_label_indices:
        return {
            'success': False,
            'error': f'Target word "{target_word}" not seen during training. '
                     f'Known words: {list(word_to_label_indices.keys())[:10]}...'
        }

    # ── Step 2: Insert [TGT] Markers ──────────────────────────────
    marked = insert_tgt_markers(sentence, target_word)

    # ── Step 3: Tokenize ───────────────────────────────────────────
    encoding = tokenizer(
        marked,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt',
    )

    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # ── Step 4: Forward Pass ───────────────────────────────────────
    model.eval()
    logits = model(input_ids, attention_mask)  # (1, NUM_CLASSES)
    logits = logits.squeeze(0)                  # (NUM_CLASSES,)

    # ── Step 5: FILTER to word-specific labels ─────────────────────
    # This is the KEY FIX: only consider senses belonging to target_word
    word_label_indices = word_to_label_indices[target_word]  # e.g. [3, 7, 12, 15]
    word_logits = logits[word_label_indices]                  # Only these logits

    # ── Step 6: Softmax over word-specific logits ──────────────────
    word_probs = torch.softmax(word_logits, dim=-1).cpu().numpy()

    # ── Step 7: Get prediction ─────────────────────────────────────
    best_local_idx   = int(np.argmax(word_probs))
    predicted_global = word_label_indices[best_local_idx]
    confidence       = float(word_probs[best_local_idx])

    # ── Step 8: Build top candidates list ─────────────────────────
    sorted_local = np.argsort(word_probs)[::-1]
    top_candidates = [
        {
            'label':      word_label_indices[i],
            'word_sense': label_to_sense[word_label_indices[i]]['word_sense'],
            'gloss':      label_to_sense[word_label_indices[i]]['gloss'],
            'sense_id':   label_to_sense[word_label_indices[i]]['sense_id'],
            'probability': float(word_probs[i]),
        }
        for i in sorted_local[:5]  # Top 5
    ]

    return {
        'success':            True,
        'sentence':           sentence,
        'target_word':        target_word,
        'marked_sentence':    marked,
        'predicted_label':    predicted_global,
        'predicted_gloss':    label_to_sense[predicted_global]['gloss'],
        'predicted_sense_id': label_to_sense[predicted_global]['sense_id'],
        'predicted_word_sense': label_to_sense[predicted_global]['word_sense'],
        'confidence':         confidence,
        'top_candidates':     top_candidates,
    }


def print_prediction(result: dict):
    """Pretty-print a prediction result."""
    print('\n' + '═' * 65)
    if not result['success']:
        print(f'  ❌ PREDICTION FAILED')
        print(f'  Error: {result["error"]}')
        print('═' * 65)
        return

    print(f'  📝 Sentence:     {result["sentence"]}')
    print(f'  🏷️  Marked:       {result["marked_sentence"]}')
    print(f'  🎯 Target word:  {result["target_word"]}')
    print(f'  ✅ Predicted:    {result["predicted_word_sense"]}  (gloss: "{result["predicted_gloss"]}")')
    print(f'  📈 Confidence:   {result["confidence"]*100:.2f}%')
    print(f'\n  🏆 Top Candidates:')
    for i, cand in enumerate(result['top_candidates']):
        icon = '👑' if i == 0 else f'  {i+1}.'
        print(f'  {icon} {cand["word_sense"]:35s} → {cand["probability"]*100:.2f}%')
    print('═' * 65)


print('✅ Inference pipeline defined!')

## 🎯 Step 17: Realistic Prediction Examples

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MULTI-WORD WSD EXAMPLES
#
# These examples test the system on realistic Bangla sentences
# covering common polysemous words: জল, মাথা, মুখ, তারা
#
# The system should correctly distinguish:
#   জল  → পানি (water) vs অশ্রু (tears) vs বৃষ্টি (rain)
#   মাথা → শরীর (body part) vs নেতা (leader) vs শীর্ষ (summit)
#   মুখ  → অঙ্গ (face/mouth) vs প্রবেশদ্বার (entrance)
#   তারা → নক্ষত্র (stars) vs সর্বনাম (they)
# ─────────────────────────────────────────────────────────────────

# Define test examples covering your dataset's words
# Update these to match the actual words in your dataset
test_examples = [
    # জল (water / tears / rain / hard work)
    ('তার চোখে জল এলো।', 'জল'),             # → tears
    ('গ্লাসে জল দাও।', 'জল'),               # → water
    ('আকাশ থেকে জল পড়ছে।', 'জল'),           # → rain

    # মাথা (head/body / leader / summit)
    ('মাথা ব্যথা করছে।', 'মাথা'),            # → body part
    ('দলের মাথা হিসেবে সে কাজ করছে।', 'মাথা'), # → leader
    ('পাহাড়ের মাথায় মেঘ জমেছে।', 'মাথা'),   # → summit

    # মুখ (face/mouth / entrance)
    ('তার মুখ দেখতে সুন্দর।', 'মুখ'),         # → face
    ('নদীর মুখ বন্ধ হয়ে গেছে।', 'মুখ'),      # → mouth/entrance

    # তারা (stars / they)
    ('রাতের আকাশে তারা ফুটে উঠেছে।', 'তারা'), # → stars
    ('তারা সবাই একসাথে গেল।', 'তারা'),        # → they

    # Error case: target word missing from sentence
    ('আকাশ পরিষ্কার আছে।', 'জল'),             # → should fail gracefully
]

print('🎯 Running Multi-Word WSD Predictions:')
print('=' * 65)

for sentence, target_word in test_examples:
    result = predict_sense(
        sentence=sentence,
        target_word=target_word,
        model=model,
        tokenizer=tokenizer,
        label_to_sense=label_to_sense,
        word_to_label_indices=word_to_label_indices,
        device=DEVICE,
    )
    print_prediction(result)

## 🐛 Step 18: Debugging — Analyze Wrong Predictions

In [ ]:
# ─────────────────────────────────────────────────────────────────
# DEBUGGING: WRONG PREDICTION ANALYSIS
#
# This section finds the hardest examples:
# - Confident wrong predictions (high confidence but incorrect)
# - Systematic confusions between specific senses
#
# Understanding these patterns helps improve the dataset.
# ─────────────────────────────────────────────────────────────────

print('🐛 Analyzing wrong predictions on validation set...')

model.eval()
all_results = []

with torch.no_grad():
    for batch_start in range(0, len(val_df), CFG.BATCH_SIZE * 2):
        batch_df = val_df.iloc[batch_start:batch_start + CFG.BATCH_SIZE * 2]

        # Tokenize batch
        texts = batch_df['marked_text'].tolist()
        encoding = tokenizer(
            texts,
            max_length=CFG.MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        input_ids      = encoding['input_ids'].to(DEVICE)
        attention_mask = encoding['attention_mask'].to(DEVICE)

        logits = model(input_ids, attention_mask)  # (batch, num_classes)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()

        for i, (_, row) in enumerate(batch_df.iterrows()):
            true_label = int(row['label'])
            target_word = row['target_word']

            # Filter to word-specific labels
            if target_word in word_to_label_indices:
                word_indices = word_to_label_indices[target_word]
                word_probs   = probs[i][word_indices]
                word_probs   = word_probs / word_probs.sum()  # Re-normalize

                best_local = int(np.argmax(word_probs))
                pred_label = word_indices[best_local]
                confidence = float(word_probs[best_local])
            else:
                pred_label = int(np.argmax(probs[i]))
                confidence = float(probs[i][pred_label])

            correct = (pred_label == true_label)
            all_results.append({
                'sentence':     row['text'],
                'target_word':  target_word,
                'true_label':   true_label,
                'true_gloss':   label_to_sense.get(true_label, {}).get('gloss', '?'),
                'pred_label':   pred_label,
                'pred_gloss':   label_to_sense.get(pred_label, {}).get('gloss', '?'),
                'confidence':   confidence,
                'correct':      correct,
            })

results_df = pd.DataFrame(all_results)
wrong_df   = results_df[~results_df['correct']].sort_values('confidence', ascending=False)

print(f'\n📊 Wrong Predictions: {len(wrong_df)} / {len(results_df)} '
      f'({len(wrong_df)/len(results_df)*100:.1f}% error rate)')

print(f'\n🔴 Top-20 Confident Wrong Predictions (worst mistakes):')
print('-' * 85)
print(f'{"Sentence"[:40]:40s} {"Word":10s} {"True":15s} {"Pred":15s} {"Conf":6s}')
print('-' * 85)
for _, row in wrong_df.head(20).iterrows():
    sent = row['sentence'][:38] + '..' if len(row['sentence']) > 40 else row['sentence']
    print(
        f'{sent:40s} '
        f'{row["target_word"]:10s} '
        f'{row["true_gloss"][:13]:15s} '
        f'{row["pred_gloss"][:13]:15s} '
        f'{row["confidence"]*100:5.1f}%'
    )

# ── Common confusion pairs ─────────────────────────────────────────
print(f'\n🔄 Most Common Confusion Pairs:')
wrong_df['confusion_pair'] = wrong_df.apply(
    lambda r: f"{r['true_gloss']} → {r['pred_gloss']}", axis=1
)
confusion_counts = wrong_df['confusion_pair'].value_counts().head(10)
for pair, count in confusion_counts.items():
    print(f'   {pair:45s} ({count} times)')

## 💾 Step 19: Save Model & All Artifacts

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MODEL SAVING
#
# We save:
#   1. BanglaBERT weights + config (via save_pretrained)
#   2. Classification head weights (separately)
#   3. Tokenizer (so inference doesn't need internet)
#   4. LabelEncoder (to decode integer labels → string)
#   5. label_to_sense mapping (label int → gloss/sense_id)
#   6. word_to_label_indices (word → list of valid label indices)
# ─────────────────────────────────────────────────────────────────

os.makedirs(CFG.MODEL_SAVE_PATH, exist_ok=True)

# 1. Save BanglaBERT encoder
model.bert.save_pretrained(CFG.MODEL_SAVE_PATH)

# 2. Save classification head weights separately
torch.save(
    model.classifier.state_dict(),
    os.path.join(CFG.MODEL_SAVE_PATH, 'classifier_head.pt')
)

# 3. Save tokenizer
tokenizer.save_pretrained(CFG.MODEL_SAVE_PATH)

# 4. Save label encoder
with open(os.path.join(CFG.MODEL_SAVE_PATH, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)

# 5. Save label → sense mapping
with open(os.path.join(CFG.MODEL_SAVE_PATH, 'label_to_sense.pkl'), 'wb') as f:
    pickle.dump(label_to_sense, f)

# 6. Save word → label indices mapping
with open(os.path.join(CFG.MODEL_SAVE_PATH, 'word_to_label_indices.pkl'), 'wb') as f:
    pickle.dump(word_to_label_indices, f)

# 7. Save config for reproducibility
import json
wsd_config = {
    'model_name':   CFG.MODEL_NAME,
    'num_classes':  NUM_CLASSES,
    'max_len':      CFG.MAX_LEN,
    'dropout_rate': CFG.DROPOUT_RATE,
    'best_val_acc': best_val_acc,
    'best_epoch':   best_epoch,
    'tgt_start':    CFG.TGT_START,
    'tgt_end':      CFG.TGT_END,
    'known_words':  list(word_to_label_indices.keys()),
}
with open(os.path.join(CFG.MODEL_SAVE_PATH, 'wsd_config.json'), 'w') as f:
    json.dump(wsd_config, f, ensure_ascii=False, indent=2)

print(f'✅ Model & artifacts saved to: {CFG.MODEL_SAVE_PATH}')
print('   Saved files:')
for fname in os.listdir(CFG.MODEL_SAVE_PATH):
    size = os.path.getsize(os.path.join(CFG.MODEL_SAVE_PATH, fname))
    print(f'   📄 {fname:40s} ({size/1e6:.2f} MB)')

## 🔄 Step 20: Load & Verify Saved Model

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MODEL LOADING — Complete function for new sessions
# ─────────────────────────────────────────────────────────────────

def load_wsd_model(model_path: str, device: torch.device):
    """
    Load saved WSD model, tokenizer, and all mappings.

    Usage:
        model, tokenizer, label_to_sense, word_to_label_indices = \
            load_wsd_model('/kaggle/working/best_wsd_model', DEVICE)

    Returns:
        (model, tokenizer, label_to_sense, word_to_label_indices)
    """
    print(f'📂 Loading WSD model from: {model_path}')

    # Load config
    with open(os.path.join(model_path, 'wsd_config.json'), 'r') as f:
        wsd_config = json.load(f)

    num_classes  = wsd_config['num_classes']
    dropout_rate = wsd_config['dropout_rate']

    # Load tokenizer
    loaded_tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)

    # Reconstruct model architecture
    loaded_model = BanglaBERTWSD(
        model_name=model_path,
        num_classes=num_classes,
        dropout_rate=dropout_rate,
    )

    # Load classification head weights
    classifier_path = os.path.join(model_path, 'classifier_head.pt')
    loaded_model.classifier.load_state_dict(
        torch.load(classifier_path, map_location=device)
    )

    loaded_model = loaded_model.to(device)
    loaded_model.eval()

    # Load mappings
    with open(os.path.join(model_path, 'label_to_sense.pkl'), 'rb') as f:
        loaded_l2s = pickle.load(f)

    with open(os.path.join(model_path, 'word_to_label_indices.pkl'), 'rb') as f:
        loaded_w2li = pickle.load(f)

    print(f'✅ Loaded! num_classes={num_classes}, known_words={len(loaded_w2li)}')
    return loaded_model, loaded_tokenizer, loaded_l2s, loaded_w2li


# ── Test reloading ─────────────────────────────────────────────────
reloaded_model, reloaded_tokenizer, reloaded_l2s, reloaded_w2li = load_wsd_model(
    CFG.MODEL_SAVE_PATH, DEVICE
)

# Quick sanity check
test_result = predict_sense(
    sentence='তার চোখে জল এলো।',
    target_word='জল',
    model=reloaded_model,
    tokenizer=reloaded_tokenizer,
    label_to_sense=reloaded_l2s,
    word_to_label_indices=reloaded_w2li,
    device=DEVICE,
)
print('\n🔬 Sanity check after reload:')
print_prediction(test_result)

## 📝 Step 21: Final Summary

In [ ]:
print('\n' + '█' * 72)
print('  BANGLA WSD — TRAINING SUMMARY')
print('█' * 72)
print(f'  Model:                {CFG.MODEL_NAME}')
print(f'  Dataset size:         {len(df)} examples')
print(f'  Target words:         {df["target_word"].nunique()}')
print(f'  Unique word-senses:   {NUM_CLASSES}')
print(f'  Train / Val split:    {len(train_df)} / {len(val_df)}')
print(f'')
print(f'  Best val accuracy:    {best_val_acc*100:.2f}% (epoch {best_epoch})')
print(f'  Final precision:      {precision*100:.2f}%')
print(f'  Final recall:         {recall*100:.2f}%')
print(f'  Final F1:             {f1*100:.2f}%')
print(f'')
print(f'  KEY FIXES APPLIED:')
print(f'  ✅ Word-specific labels (জল_অশ্রু, মাথা_শরীর, etc.)')
print(f'  ✅ [TGT] ... [/TGT] markers in every training sentence')
print(f'  ✅ Inference filters label space to target_word only')
print(f'  ✅ Input validation: rejects missing target_word')
print(f'  ✅ Stratified split (no class leakage)')
print(f'  ✅ Duplicate removal before split')
print(f'  ✅ Early stopping (patience={CFG.PATIENCE})')
print(f'  ✅ Dropout={CFG.DROPOUT_RATE} + weight decay={CFG.WEIGHT_DECAY}')
print(f'  ✅ Gradient clipping (max norm={CFG.MAX_GRAD_NORM})')
print(f'  ✅ Weighted CrossEntropyLoss for class imbalance')
print(f'  ✅ Discriminative LR (BERT vs classifier head)')
print(f'  ✅ Linear warmup LR schedule')
print(f'  ✅ No tokenizer.add_tokens() — no embedding resize')
print(f'  ✅ TOKENIZERS_PARALLELISM=false')
print('█' * 72)

In [ ]:
# Show all unique target words

all_target_words = df['target_word'].unique()

print("✅ All target words:\n")

for i, word in enumerate(all_target_words, 1):
    print(f"{i}. {word}")

print(f"\n📌 Total target words: {len(all_target_words)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# AUTOMATIC RANDOM WSD TESTING
# Tests random words from the dataset
# ─────────────────────────────────────────────────────────────

import random

# Number of random words to test
NUM_TESTS = 20

# Get all target words
all_words = df['target_word'].unique().tolist()

# Randomly select words
random_words = random.sample(all_words, NUM_TESTS)

print("🧠 RANDOM MULTI-WORD WSD TESTING")
print("=" * 80)

for idx, word in enumerate(random_words, 1):

    print(f"\n🔹 TEST #{idx}")
    print("=" * 80)

    # Get samples for this word
    word_samples = df[df['target_word'] == word]

    # Random sample sentence
    sample_row = word_samples.sample(1).iloc[0]

    sentence = sample_row['text']

    try:

        result = predict_sense(
            sentence=sentence,
            target_word=word,
            model=model,
            tokenizer=tokenizer,
            label_to_sense=label_to_sense,
            word_to_label_indices=word_to_label_indices,
            device=DEVICE,
        )

        print_prediction(result)

    except Exception as e:
        print(f"❌ Error testing word '{word}': {e}")

print("\n✅ RANDOM TESTING COMPLETE!")

In [ ]:
# Dataset summary

summary = df.groupby('target_word')['sense_gloss'].nunique()

print(summary)

In [ ]:
# ─────────────────────────────────────────────
# Show all senses for each target word
# ─────────────────────────────────────────────

grouped = df.groupby('target_word')['sense_gloss'].unique()

for word, senses in grouped.items():

    print("=" * 60)
    print(f"🎯 Target Word: {word}")
    print(f"📌 Total Senses: {len(senses)}\n")

    for i, sense in enumerate(senses, 1):
        print(f"{i}. {sense}")

    print()

# MFS Baseline

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 📊 MFS BASELINE — Most Frequent Sense
# সবচেয়ে সহজ baseline: training-এ যে sense সবচেয়ে বেশি
# সেটাই সবসময় predict করো। কোনো model নেই।
# ─────────────────────────────────────────────────────────────────

from collections import Counter
from sklearn.metrics import f1_score, accuracy_score

# ── Step 1: Training data থেকে প্রতিটি শব্দের MFS বের করো ────────
mfs_dict = {}
for word, group in train_df.groupby('target_word'):
    # সবচেয়ে বেশিবার দেখা label
    most_common_label = group['label'].value_counts().idxmax()
    mfs_dict[word] = most_common_label

print(f"✅ MFS built for {len(mfs_dict)} words")
print("\n📋 Sample MFS predictions:")
for word, lbl in list(mfs_dict.items())[:5]:
    gloss = label_to_sense[lbl]['gloss']
    print(f"   {word:15s} → always predicts: '{gloss}'")

# ── Step 2: Validation set-এ MFS দিয়ে predict করো ────────────────
mfs_preds  = []
mfs_labels = []

for _, row in val_df.iterrows():
    word = row['target_word']
    if word in mfs_dict:
        mfs_preds.append(mfs_dict[word])
        mfs_labels.append(row['label'])

# ── Step 3: Metrics হিসাব করো ─────────────────────────────────────
mfs_acc = accuracy_score(mfs_labels, mfs_preds)
mfs_f1  = f1_score(mfs_labels, mfs_preds, average='weighted')
mfs_f1_macro = f1_score(mfs_labels, mfs_preds, average='macro')

print(f"\n{'='*50}")
print(f"  📊 MFS BASELINE RESULTS")
print(f"{'='*50}")
print(f"  Accuracy          : {mfs_acc*100:.2f}%")
print(f"  F1 (weighted)     : {mfs_f1*100:.2f}%")
print(f"  F1 (macro)        : {mfs_f1_macro*100:.2f}%")
print(f"{'='*50}")

# ── Step 4: BanglaBERT-এর সাথে তুলনা ────────────────────────────
banglabert_f1  = 86.63   # তোমার trained model-এর result
banglabert_acc = 88.09

print(f"\n📈 COMPARISON TABLE")
print(f"{'─'*55}")
print(f"  {'Model':<20} {'Accuracy':>10} {'F1 (weighted)':>15}")
print(f"{'─'*55}")
print(f"  {'MFS Baseline':<20} {mfs_acc*100:>9.2f}% {mfs_f1*100:>14.2f}%")
print(f"  {'BanglaBERT (ours)':<20} {banglabert_acc:>9.2f}% {banglabert_f1:>14.2f}%")
print(f"{'─'*55}")
improvement_f1  = banglabert_f1  - mfs_f1*100
improvement_acc = banglabert_acc - mfs_acc*100
print(f"  {'Improvement':<20} {improvement_acc:>+9.2f}% {improvement_f1:>+14.2f}%")
print(f"{'─'*55}")

if improvement_f1 > 15:
    print("\n✅ BanglaBERT significantly outperforms MFS baseline.")
    print("   This proves the model learned meaningful patterns.")
elif improvement_f1 > 5:
    print("\n✅ BanglaBERT moderately outperforms MFS baseline.")
else:
    print("\n⚠️  Small improvement — dataset may be imbalanced.")

# mBERT Comparison

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🌐 mBERT COMPARISON
# Google-এর Multilingual BERT দিয়ে same dataset-এ train করো
# তারপর BanglaBERT-এর সাথে তুলনা করো
# ─────────────────────────────────────────────────────────────────

print("Loading mBERT tokenizer & model...")

MBERT_NAME      = 'bert-base-multilingual-cased'
mbert_tokenizer = AutoTokenizer.from_pretrained(MBERT_NAME)

# ── mBERT-এর জন্য আলাদা Dataset বানাও ───────────────────────────
class mBERTDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=CFG.MAX_LEN):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        encoding = self.tokenizer(
            row['marked_text'],
            max_length  = self.max_len,
            padding     = 'max_length',
            truncation  = True,
            return_tensors = 'pt',
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(0),
            'attention_mask' : encoding['attention_mask'].squeeze(0),
            'labels'         : torch.tensor(row['label'], dtype=torch.long),
        }

mbert_train_dataset = mBERTDataset(train_df, mbert_tokenizer)
mbert_val_dataset   = mBERTDataset(val_df,   mbert_tokenizer)

mbert_train_loader  = DataLoader(
    mbert_train_dataset, batch_size=CFG.BATCH_SIZE,
    shuffle=True,  num_workers=2
)
mbert_val_loader    = DataLoader(
    mbert_val_dataset, batch_size=CFG.BATCH_SIZE,
    shuffle=False, num_workers=2
)
print("✅ mBERT datasets ready!")

# ── mBERT Model (same architecture, different backbone) ───────────
class mBERTWSD(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        from transformers import BertModel
        self.bert       = BertModel.from_pretrained(MBERT_NAME)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # [CLS] token
        return self.classifier(self.dropout(cls))

mbert_model = mBERTWSD(num_classes=NUM_CLASSES).to(DEVICE)
print(f"✅ mBERT model loaded → {MBERT_NAME}")

# ── Training loop (same as BanglaBERT) ───────────────────────────
MBERT_EPOCHS = 10   # কম epoch — faster comparison
mbert_optimizer = AdamW(
    mbert_model.parameters(),
    lr=2e-5, weight_decay=0.01
)
mbert_scheduler = get_linear_schedule_with_warmup(
    mbert_optimizer,
    num_warmup_steps   = len(mbert_train_loader),
    num_training_steps = MBERT_EPOCHS * len(mbert_train_loader)
)

# Class weights (same as before)
class_weights_np = compute_class_weight(
    'balanced',
    classes = np.unique(train_df['label'].values),
    y       = train_df['label'].values
)
mbert_criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights_np, dtype=torch.float32).to(DEVICE)
)

# ── Train mBERT ───────────────────────────────────────────────────
best_mbert_f1  = 0.0
best_mbert_acc = 0.0
print(f"\n🚀 Training mBERT for {MBERT_EPOCHS} epochs...")
print("─" * 50)

for epoch in range(1, MBERT_EPOCHS + 1):
    # Train
    mbert_model.train()
    for batch in mbert_train_loader:
        mbert_optimizer.zero_grad()
        logits = mbert_model(
            batch['input_ids'].to(DEVICE),
            batch['attention_mask'].to(DEVICE)
        )
        loss = mbert_criterion(logits, batch['labels'].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mbert_model.parameters(), 1.0)
        mbert_optimizer.step()
        mbert_scheduler.step()

    # Validate
    mbert_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in mbert_val_loader:
            logits = mbert_model(
                batch['input_ids'].to(DEVICE),
                batch['attention_mask'].to(DEVICE)
            )
            preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            labels.extend(batch['labels'].cpu().numpy())

    epoch_f1  = f1_score(labels, preds, average='weighted') * 100
    epoch_acc = accuracy_score(labels, preds) * 100

    if epoch_f1 > best_mbert_f1:
        best_mbert_f1  = epoch_f1
        best_mbert_acc = epoch_acc

    print(f"  Epoch {epoch:02d}/{MBERT_EPOCHS} → "
          f"Acc: {epoch_acc:.2f}%  F1: {epoch_f1:.2f}%"
          + (" ⭐" if epoch_f1 >= best_mbert_f1 else ""))

# ── Final Comparison Table ────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  🏆 FINAL MODEL COMPARISON")
print(f"{'='*60}")
print(f"  {'Model':<25} {'Accuracy':>10} {'F1 (weighted)':>15}")
print(f"{'─'*60}")
print(f"  {'MFS Baseline':<25} {mfs_acc*100:>9.2f}% {mfs_f1*100:>14.2f}%")
print(f"  {'mBERT':<25} {best_mbert_acc:>9.2f}% {best_mbert_f1:>14.2f}%")
print(f"  {'BanglaBERT (ours)':<25} {banglabert_acc:>9.2f}% {banglabert_f1:>14.2f}%")
print(f"{'='*60}")

# Winner
if banglabert_f1 > best_mbert_f1:
    diff = banglabert_f1 - best_mbert_f1
    print(f"\n✅ BanglaBERT beats mBERT by {diff:.2f}% F1")
    print("   ✍️  Thesis claim: Language-specific pretraining")
    print("       outperforms multilingual model for Bangla WSD.")
else:
    diff = best_mbert_f1 - banglabert_f1
    print(f"\n⚠️  mBERT beats BanglaBERT by {diff:.2f}% F1")
    print("   ✍️  Thesis claim: Multilingual models can be")
    print("       competitive for low-resource WSD tasks.")

 # XLM-R

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🌍 XLM-R COMPARISON
# ─────────────────────────────────────────────────────────────────

XLMR_NAME      = 'xlm-roberta-base'
xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_NAME)

# Dataset (same structure, different tokenizer)
xlmr_train_dataset = mBERTDataset(train_df, xlmr_tokenizer)
xlmr_val_dataset   = mBERTDataset(val_df,   xlmr_tokenizer)

xlmr_train_loader = DataLoader(
    xlmr_train_dataset, batch_size=CFG.BATCH_SIZE,
    shuffle=True, num_workers=2
)
xlmr_val_loader = DataLoader(
    xlmr_val_dataset, batch_size=CFG.BATCH_SIZE,
    shuffle=False, num_workers=2
)

# Model
class XLMRWSD(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        from transformers import XLMRobertaModel
        self.bert       = XLMRobertaModel.from_pretrained(XLMR_NAME)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))

xlmr_model     = XLMRWSD(num_classes=NUM_CLASSES).to(DEVICE)
xlmr_optimizer = AdamW(xlmr_model.parameters(), lr=2e-5, weight_decay=0.01)
xlmr_scheduler = get_linear_schedule_with_warmup(
    xlmr_optimizer,
    num_warmup_steps   = len(xlmr_train_loader),
    num_training_steps = 10 * len(xlmr_train_loader)
)
xlmr_criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights_np, dtype=torch.float32).to(DEVICE)
)

best_xlmr_f1, best_xlmr_acc = 0.0, 0.0
print(f"🚀 Training XLM-R for 10 epochs...")
print("─" * 50)

for epoch in range(1, 11):
    xlmr_model.train()
    for batch in xlmr_train_loader:
        xlmr_optimizer.zero_grad()
        logits = xlmr_model(
            batch['input_ids'].to(DEVICE),
            batch['attention_mask'].to(DEVICE)
        )
        loss = xlmr_criterion(logits, batch['labels'].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(xlmr_model.parameters(), 1.0)
        xlmr_optimizer.step()
        xlmr_scheduler.step()

    xlmr_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in xlmr_val_loader:
            logits = xlmr_model(
                batch['input_ids'].to(DEVICE),
                batch['attention_mask'].to(DEVICE)
            )
            preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            labels.extend(batch['labels'].cpu().numpy())

    epoch_f1  = f1_score(labels, preds, average='weighted') * 100
    epoch_acc = accuracy_score(labels, preds) * 100
    if epoch_f1 > best_xlmr_f1:
        best_xlmr_f1  = epoch_f1
        best_xlmr_acc = epoch_acc
    print(f"  Epoch {epoch:02d}/10 → Acc: {epoch_acc:.2f}%  F1: {epoch_f1:.2f}%"
          + (" ⭐" if epoch_f1 >= best_xlmr_f1 else ""))

# ── Complete Final Table (সব model একসাথে) ────────────────────────
print(f"\n{'='*62}")
print(f"  🏆 COMPLETE MODEL COMPARISON — BANGLA WSD")
print(f"{'='*62}")
print(f"  {'Model':<25} {'Accuracy':>10} {'F1 (weighted)':>15}")
print(f"{'─'*62}")
print(f"  {'MFS Baseline':<25} {mfs_acc*100:>9.2f}% {mfs_f1*100:>14.2f}%")
print(f"  {'mBERT':<25} {best_mbert_acc:>9.2f}% {best_mbert_f1:>14.2f}%")
print(f"  {'XLM-R Base':<25} {best_xlmr_acc:>9.2f}% {best_xlmr_f1:>14.2f}%")
print(f"  {'BanglaBERT (ours)':<25} {banglabert_acc:>9.2f}% {banglabert_f1:>14.2f}%")
print(f"{'='*62}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🌍 XLM-R COMPARISON (FIXED)
# ─────────────────────────────────────────────────────────────────

XLMR_NAME      = 'xlm-roberta-base'
xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_NAME)

# ── Dataset ───────────────────────────────────────────────────────
xlmr_train_dataset = mBERTDataset(train_df, xlmr_tokenizer)
xlmr_val_dataset   = mBERTDataset(val_df,   xlmr_tokenizer)

xlmr_train_loader = DataLoader(
    xlmr_train_dataset, batch_size=CFG.BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True
)
xlmr_val_loader = DataLoader(
    xlmr_val_dataset, batch_size=CFG.BATCH_SIZE * 2,
    shuffle=False, num_workers=2, pin_memory=True
)

# ── Model ─────────────────────────────────────────────────────────
class XLMRWSD(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        from transformers import XLMRobertaModel
        self.bert       = XLMRobertaModel.from_pretrained(XLMR_NAME)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))

xlmr_model = XLMRWSD(num_classes=NUM_CLASSES).to(DEVICE)

# ── Optimizer — discriminative LR (same as BanglaBERT) ───────────
XLMR_EPOCHS  = 10
xlmr_optimizer = AdamW(
    [
        {'params': xlmr_model.bert.parameters(),       'lr': CFG.LEARNING_RATE},
        {'params': xlmr_model.classifier.parameters(), 'lr': CFG.LEARNING_RATE * 10},
    ],
    weight_decay=CFG.WEIGHT_DECAY,
)

total_steps  = len(xlmr_train_loader) * XLMR_EPOCHS
warmup_steps = int(total_steps * CFG.WARMUP_RATIO)

xlmr_scheduler = get_linear_schedule_with_warmup(
    xlmr_optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)

# ── Loss — correct variable name: class_weights ──────────────────
xlmr_criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor   # ← class_weights_tensor (তোমার notebook-এর আসল নাম)
)

# ── Training loop ─────────────────────────────────────────────────
best_xlmr_f1, best_xlmr_acc = 0.0, 0.0
print(f"🚀 Training XLM-R for {XLMR_EPOCHS} epochs...")
print("─" * 50)

for epoch in range(1, XLMR_EPOCHS + 1):
    # Train
    xlmr_model.train()
    total_loss = 0
    for batch in xlmr_train_loader:
        xlmr_optimizer.zero_grad()
        logits = xlmr_model(
            batch['input_ids'].to(DEVICE),
            batch['attention_mask'].to(DEVICE)
        )
        loss = xlmr_criterion(logits, batch['labels'].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(xlmr_model.parameters(), 1.0)
        xlmr_optimizer.step()
        xlmr_scheduler.step()
        total_loss += loss.item()

    # Validate
    xlmr_model.eval()
    preds, true_labels = [], []
    with torch.no_grad():
        for batch in xlmr_val_loader:
            logits = xlmr_model(
                batch['input_ids'].to(DEVICE),
                batch['attention_mask'].to(DEVICE)
            )
            preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            true_labels.extend(batch['labels'].cpu().numpy())

    epoch_f1  = f1_score(true_labels, preds, average='weighted') * 100
    epoch_acc = accuracy_score(true_labels, preds) * 100
    avg_loss  = total_loss / len(xlmr_train_loader)

    is_best = epoch_f1 > best_xlmr_f1
    if is_best:
        best_xlmr_f1  = epoch_f1
        best_xlmr_acc = epoch_acc

    print(f"  Epoch {epoch:02d}/{XLMR_EPOCHS} → "
          f"Loss: {avg_loss:.4f}  "
          f"Acc: {epoch_acc:.2f}%  "
          f"F1: {epoch_f1:.2f}%"
          + (" ⭐" if is_best else ""))

# ── Complete Final Table ───────────────────────────────────────────
print(f"\n{'='*62}")
print(f"  🏆 COMPLETE MODEL COMPARISON — BANGLA WSD")
print(f"{'='*62}")
print(f"  {'Model':<25} {'Accuracy':>10} {'F1 (weighted)':>15}")
print(f"{'─'*62}")
print(f"  {'MFS Baseline':<25} {mfs_acc*100:>9.2f}% {mfs_f1*100:>14.2f}%")
print(f"  {'mBERT':<25} {best_mbert_acc:>9.2f}% {best_mbert_f1:>14.2f}%")
print(f"  {'XLM-R Base':<25} {best_xlmr_acc:>9.2f}% {best_xlmr_f1:>14.2f}%")
print(f"  {'BanglaBERT (ours)':<25} {banglabert_acc:>9.2f}% {banglabert_f1:>14.2f}%")
print(f"{'='*62}")

# ── Visualization ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

models     = ['MFS\nBaseline', 'mBERT', 'XLM-R\nBase', 'BanglaBERT\n(Ours)']
acc_vals   = [mfs_acc*100, best_mbert_acc, best_xlmr_acc, banglabert_acc]
f1_vals    = [mfs_f1*100,  best_mbert_f1,  best_xlmr_f1,  banglabert_f1]
colors     = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c']

x     = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar(x - width/2, acc_vals, width, label='Accuracy',
               color=colors, alpha=0.9)
bars2 = ax.bar(x + width/2, f1_vals,  width, label='F1 (weighted)',
               color=colors, alpha=0.55, hatch='//')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.8,
            f'{bar.get_height():.1f}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Bangla WSD — Model Comparison\n(Accuracy & Weighted F1)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0, 105)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=banglabert_f1, color='red',
           linestyle='--', alpha=0.4, linewidth=1.5)
ax.text(3.6, banglabert_f1 + 1,
        f'BanglaBERT F1={banglabert_f1:.1f}%',
        color='red', fontsize=8, alpha=0.7)

plt.tight_layout()
plt.savefig('/kaggle/working/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Chart saved → /kaggle/working/model_comparison.png")
print("   ✍️  এই figure সরাসরি thesis-এ ব্যবহার করো।")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 📊 FINAL COMPARISON CHART — Thesis Figure
# ─────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import numpy as np

models  = ['MFS\nBaseline', 'mBERT', 'XLM-R\nBase', 'BanglaBERT\n(Ours)']
acc_scores = [mfs_acc*100, best_mbert_acc, best_xlmr_acc, banglabert_acc]
f1_scores  = [mfs_f1*100,  best_mbert_f1,  best_xlmr_f1,  banglabert_f1]

x     = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, acc_scores, width,
               label='Accuracy', color=['#95a5a6','#3498db','#2ecc71','#e74c3c'],
               alpha=0.85)
bars2 = ax.bar(x + width/2, f1_scores, width,
               label='F1 (weighted)', color=['#95a5a6','#3498db','#2ecc71','#e74c3c'],
               alpha=0.5, hatch='//')

# Value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Bangla WSD — Model Comparison\n(Accuracy & Weighted F1)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0, 100)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=banglabert_f1, color='red', linestyle='--', alpha=0.4,
           label=f'BanglaBERT F1 = {banglabert_f1:.1f}%')

plt.tight_layout()
plt.savefig('/kaggle/working/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Chart saved → /kaggle/working/model_comparison.png")
